In [ ]:
import os
import subprocess

DATA_DIR = "data"
if not os.path.exists(DATA_DIR):
    os.mkdir(DATA_DIR)


for month in range(1, 13):
    if not os.path.exists(f"{DATA_DIR}/{month}.parquet"):
        subprocess.run(
            [
                "wget",
                f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-{month:02}.parquet",
                "-O",
                f"{DATA_DIR}/{month}.parquet",
            ]
        )


total_size = sum(
    os.path.getsize(f"{DATA_DIR}/{month}.parquet") for month in range(1, 13)
)  # bytes
total_size_mb = total_size // (1024 * 1024)
print(f"Total dataset size: {total_size_mb} MB")

Total dataset size: 660 MB


In [ ]:
if not os.path.exists(f"{DATA_DIR}/taxi_zone_lookup.csv"):
    subprocess.run(
        [
            "wget",
            "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
            "-O",
            f"{DATA_DIR}/taxi_zone_lookup.csv",
        ]
    )

### 1. Data reading
- load all 2024 months of taxi rides
- also load taxi zone lookup data
- include only rides starting in 2024 and ending at most at 01.01.2025
- optimize data types, particularly for integers and categorical strings

In [3]:
import polars as pl


dfs = []
for month in range(1, 13):
    lf = pl.scan_parquet(f"{DATA_DIR}/{month}.parquet")
    lf = lf.with_columns(
        pl.col("tpep_pickup_datetime").dt.cast_time_unit("ms"),
        pl.col("tpep_dropoff_datetime").dt.cast_time_unit("ms"),
        pl.col("VendorID", "RatecodeID", "payment_type", "passenger_count").cast(
            pl.UInt8
        ),
        pl.col("PULocationID", "DOLocationID").cast(pl.UInt16),
        pl.col("store_and_fwd_flag").cast(pl.Categorical),
    )
    dfs.append(lf)

lf_2024 = pl.concat(dfs)
print(lf_2024.explain())

UNION[maintain_order: true]
  PLAN 0:
     WITH_COLUMNS:
     [col("tpep_pickup_datetime").dt.cast_time_unit(), col("tpep_dropoff_datetime").dt.cast_time_unit(), col("VendorID").strict_cast(UInt8), col("RatecodeID").strict_cast(UInt8), col("payment_type").strict_cast(UInt8), col("passenger_count").strict_cast(UInt8), col("PULocationID").strict_cast(UInt16), col("DOLocationID").strict_cast(UInt16), col("store_and_fwd_flag").strict_cast(Categorical)] 
      Parquet SCAN [data/1.parquet]
      PROJECT */19 COLUMNS
      ESTIMATED ROWS: 2964624
  PLAN 1:
     WITH_COLUMNS:
     [col("tpep_pickup_datetime").dt.cast_time_unit(), col("tpep_dropoff_datetime").dt.cast_time_unit(), col("VendorID").strict_cast(UInt8), col("RatecodeID").strict_cast(UInt8), col("payment_type").strict_cast(UInt8), col("passenger_count").strict_cast(UInt8), col("PULocationID").strict_cast(UInt16), col("DOLocationID").strict_cast(UInt16), col("store_and_fwd_flag").strict_cast(Categorical)] 
      Parquet SCAN [data/2.

In [ ]:
from datetime import datetime

lf_2024 = lf_2024.filter(
    pl.col("tpep_pickup_datetime") >= datetime(2024, 1, 1),
    pl.col("tpep_pickup_datetime") < datetime(2025, 1, 1),
    pl.col("tpep_dropoff_datetime") <= datetime(2025, 1, 1),
)

In [ ]:
zones = pl.read_csv(f"{DATA_DIR}/taxi_zone_lookup.csv")

In [6]:
zones.describe()

statistic,LocationID,Borough,Zone,service_zone
str,f64,str,str,str
"""count""",265.0,"""265""","""265""","""265"""
"""null_count""",0.0,"""0""","""0""","""0"""
"""mean""",133.0,null,null,null
"""std""",76.643112,null,null,null
"""min""",1.0,"""Bronx""","""Allerton/Pelham Gardens""","""Airports"""
"""25%""",67.0,null,null,null
"""50%""",133.0,null,null,null
"""75%""",199.0,null,null,null
"""max""",265.0,"""Unknown""","""Yorkville West""","""Yellow Zone"""


### 2. Data cleaning and filtering
- fill NULL values in passengers_count with 1
- remove rides with zero passengers
- if a ride has over 6 passengers, replace the value with 6
- remove rides lasting over 2 hours
- in all money-related columns, replace values with absolute value to fix - - negative amounts
- remove rides with over 1000 dollars in any money-related column
- remove rows with values of RatecodeID or VendorID missing or outside those - defined in the data dictionary

In [ ]:
money_cols = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
]

lf_2024 = (
    lf_2024.with_columns(pl.col("passenger_count").fill_null(1))
    .filter(pl.col("passenger_count") > 0)
    .with_columns(pl.col("passenger_count").clip(upper_bound=6))
    .filter(
        (pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
        <= pl.duration(hours=2)
    )
    .with_columns(pl.col(money_cols).abs())
    .filter(pl.max_horizontal(pl.col(money_cols)) <= 1000)
    .filter(
        pl.col("RatecodeID").is_in([1, 2, 3, 4, 5, 6, 99]),
        pl.col("VendorID").is_in([1, 2, 6, 7]),
    )
)

3. Data transformation
- combine payment type values for "No charge", "Dispute", "Unknown" and - - - "Voided trip" into one type, so we have only credit card, cash, or other
- replace PULocationID and DOLocationID with borough names by joining taxi - - zone lookup and removing unnecessary columns
- add boolean variable is_airport_ride, true if there is non-zero airport fee
- add boolean variable is_rush_hour, rush hours are defined as 6:30–9:30 and 15:30-20:00 during weekdays

0 = Flex Fare trip 1 = Credit card 2 = Cash 3 = No charge 4 = Dispute 5 = Unknown 6 = Voided trip

In [ ]:
zones_lf = zones.lazy().select(
    pl.col("LocationID").cast(pl.UInt16), pl.col("Borough").cast(pl.Categorical)
)

time_of_day = pl.col("tpep_pickup_datetime").dt.time()

lf_2024 = (
    lf_2024.with_columns(
        pl.when(pl.col("payment_type") == 1)
        .then(pl.lit("credit_card"))
        .when(pl.col("payment_type") == 2)
        .then(pl.lit("cash"))
        .otherwise(pl.lit("other"))
        .cast(pl.Categorical)
        .alias("payment_type")
    )
    .join(
        zones_lf.rename({"LocationID": "PULocationID", "Borough": "pu_borough"}),
        on="PULocationID",
        how="left",
    )
    .join(
        zones_lf.rename({"LocationID": "DOLocationID", "Borough": "do_borough"}),
        on="DOLocationID",
        how="left",
    )
    .drop("PULocationID", "DOLocationID")
    .with_columns(
        is_airport_ride=pl.col("Airport_fee") > 0,
        is_rush_hour=(pl.col("tpep_pickup_datetime").dt.weekday() <= 5)
        & (
            time_of_day.is_between(pl.time(6, 30), pl.time(9, 30), closed="left")
            | time_of_day.is_between(pl.time(15, 30), pl.time(20, 0), closed="left")
        ),
    )
)

4. Feature extraction
- apply dummy encoding to features:
    - payment type
    - pickup borough
    - dropoff borough


In [9]:
print(zones.select("Borough").unique())

shape: (8, 1)
┌───────────────┐
│ Borough       │
│ ---           │
│ str           │
╞═══════════════╡
│ Staten Island │
│ Unknown       │
│ Brooklyn      │
│ Bronx         │
│ Queens        │
│ N/A           │
│ EWR           │
│ Manhattan     │
└───────────────┘


In [10]:
payment_types = ["credit_card", "cash", "other"]
boroughs = zones["Borough"].unique().sort().to_list()


def slug(s: str):
    return s.lower().replace(" ", "_").replace("/", "_")


dummy_expression = (
    [
        (pl.col("payment_type") == p).cast(pl.UInt8).alias(f"payment_{p}")
        for p in payment_types
    ]
    + [
        (pl.col("pu_borough") == b).cast(pl.UInt8).alias(f"pu_{slug(b)}")
        for b in boroughs
    ]
    + [
        (pl.col("do_borough") == b).cast(pl.UInt8).alias(f"do_{slug(b)}")
        for b in boroughs
    ]
)

lf_2024 = lf_2024.with_columns(dummy_expression)

In [ ]:
print(lf_2024.collect_schema().names())

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'pu_borough', 'do_borough', 'is_airport_ride', 'is_rush_hour', 'date', 'payment_credit_card', 'payment_cash', 'payment_other', 'pu_bronx', 'pu_brooklyn', 'pu_ewr', 'pu_manhattan', 'pu_n_a', 'pu_queens', 'pu_staten_island', 'pu_unknown', 'do_bronx', 'do_brooklyn', 'do_ewr', 'do_manhattan', 'do_n_a', 'do_queens', 'do_staten_island', 'do_unknown']


- add integer variables for counting daily events:
    - total number of rides (target variable)
    - number of airport rides
    - number of rush hour rides


In [12]:
counts_exprs = [
    pl.len().alias("n_rides"),
    pl.col("is_airport_ride").sum().alias("n_airport_rides"),
    pl.col("is_rush_hour").sum().alias("n_rush_hour_rides"),
]

- add features aggregating daily rides information:
    - average fare amount
    - median distance
    - sum of total amounts
    - total amount paid by card, cash, and other
    - total congestion surcharge
    - total number of passengers


In [13]:
aggregate_exprs = [
    pl.col("fare_amount").mean().alias("avg_fare_amount"),
    pl.col("trip_distance").median().alias("median_distance"),
    pl.col("total_amount").sum().alias("sum_total_amount"),
    pl.col("total_amount")
    .filter(pl.col("payment_type") == "credit_card")
    .sum()
    .alias("total_amount_card"),
    pl.col("total_amount")
    .filter(pl.col("payment_type") == "cash")
    .sum()
    .alias("total_amount_cash"),
    pl.col("total_amount")
    .filter(pl.col("payment_type") == "other")
    .sum()
    .alias("total_amount_other"),
    pl.col("congestion_surcharge").sum().alias("total_congestion_surcharge"),
    pl.col("passenger_count").sum().alias("total_passengers"),
]

- add time features:
    - quarter
    - month
    - day_of_month
    - day_of_week
    - is_weekend (boolean)
- add column date, indicating day with given features
- make sure to exclude other columns, unnecessary for machine learning, e.g. - IDs, timestamps, unused financial information
- properly name all columns
- save results as dataset.parquet file

In [ ]:
dummy_cols = [
    e.meta.output_name() for e in dummy_expression
]  # names of the dummy columns


daily = (
    lf_2024.with_columns(date=pl.col("tpep_pickup_datetime").dt.date())
    .group_by("date")
    .agg(
        *counts_exprs,
        *aggregate_exprs,
        pl.col(dummy_cols).sum(),
    )
)

In [ ]:
daily = daily.with_columns(
    quarter=pl.col("date").dt.quarter(),
    month=pl.col("date").dt.month(),
    day_of_month=pl.col("date").dt.day(),
    day_of_week=pl.col("date").dt.weekday(),
    is_weekend=pl.col("date").dt.weekday() > 5,
).sort("date")

In [ ]:
dataset = daily.collect()
dataset.write_parquet("dataset.parquet")